In [ ]:
import sys; sys.path.insert(0, '../src')
import dst_models as dst_eqs
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt

from dst_models import burton_dst
from matplotlib.ticker import FixedLocator

In [ ]:
results = np.load("../results/test_result.npz", allow_pickle=True)
inp = results['inp']
tar = results['tar']
pred = results['pred']
tar_t = results['tar_t']

In [ ]:
inp.shape

In [ ]:
min_values = np.array([ 8.9166665e-01, -5.1355000e+01,  2.5678000e+02,  1.6000000e-01, 3.9999999e-02, -3.0938334e+01])
max_values = np.array([  68.475,      38.43833,  1089.08,       64.945,      65.4,        31.791666])

In [ ]:
pred = pred * (max_values[1] - min_values[1]) + min_values[1]
tar = tar * (max_values[1] - min_values[1]) + min_values[1]

In [ ]:
inp = inp * (max_values - min_values) + min_values

In [ ]:
# 2022.12.19 02:00 - 04:00  492-496
# 2024.03.03.12:00 - 13:30 774- 777
# 2024.10.10 18:30 - 19:30  951-953

In [ ]:
omni = pd.read_csv("../data/omni_hourly_2001_2025.csv")

In [ ]:
omni['time'] = pd.to_datetime(omni['time'])

In [ ]:
omni.set_index("time", inplace=True)

In [ ]:
omni.columns

In [ ]:
tar_t = pd.to_datetime(tar_t)

In [ ]:
tar_t[0]

In [ ]:
tar_window_omni = omni[tar_t[0][0]-datetime.timedelta(hours=10):tar_t[0][-1]]
tar_window_dst = tar_window_omni['DST'].values
tar_window_v = tar_window_omni['V'].values
tar_window_bz = tar_window_omni['BZ_GSM'].values
tar_window_n = tar_window_omni['N'].values

In [ ]:
time

In [ ]:
dst_preds_list = []
dst_persis_list = []
dst_om_list = []
tar_dst = []

for idx in range(len(tar_t)):
    tar_window_omni = omni[tar_t[idx][0]-datetime.timedelta(hours=10):tar_t[idx][-1]]
    tar_window_dst = tar_window_omni['DST'].values
    tar_window_v = tar_window_omni['V'].values
    tar_window_bz = tar_window_omni['BZ_GSM'].values
    tar_window_n = tar_window_omni['N'].values
    tar_dst.append(tar_window_dst[10:])

    time = np.arange(0, 12.5, 0.5)
    v = np.full_like(time, inp[idx, -1, 2])
    n = np.full_like(time, inp[idx, -1, 3])
    bz = np.append(inp[idx, -1, 1], pred[idx])
    dst_pred = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, tar_window_dst[9])
    dst_preds_list.append(dst_pred)

    time = np.arange(0, 12.5, 0.5)
    v = np.full_like(time, inp[idx, -1, 2])
    n = np.full_like(time, inp[idx, -1, 3])
    bz = np.full_like(time, inp[idx, -1, 1])
    dst_persis = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, tar_window_dst[9])
    dst_persis_list.append(dst_persis)

    time = np.arange(0, 13, 1)
    v = tar_window_v[9:]
    n = tar_window_n[9:]
    bz = tar_window_bz[9:]
    dst_om = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, tar_window_dst[9:9+16], prediction=False)
    dst_om_list.append(dst_om)

dst_preds_list = np.array(dst_preds_list)
dst_persis_list = np.array(dst_persis_list)
dst_om_list = np.array(dst_om_list)

In [ ]:
len(dst_om)

In [ ]:
len(tar_window_dst[9:9+12])

In [ ]:
dst_om_list

In [ ]:
dst_preds_list.shape

In [ ]:
dst_preds_list.reshape((-1, dst_preds_list.shape[-1]//2, 2)).mean(axis=-1)

In [ ]:
tar_dst = np.array(tar_dst)

In [ ]:
from scipy.stats import pearsonr

In [ ]:
pearsonr(x,y)[0]

In [ ]:
dst_preds_list.shape

In [ ]:
len(dst_persis_1h[0])

In [ ]:
rmse

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
dst_preds_1h = dst_preds_list[:, :-1].reshape((-1, dst_preds_list.shape[-1]//2, 2)).mean(axis=-1)

for i in range(12):
    ax = axes.flat[i]

    x = tar_dst[:, i]
    y = dst_preds_1h[:, i]

    mask = np.isfinite(x) & np.isfinite(y)
    cc = pearsonr(x[mask], y[mask])[0]
    rmse = np.mean(np.sqrt((x[mask] - y[mask]) ** 2))
    ax.scatter(
        x,
        y,
        color='k',
        marker='D',
        s=20
    )

    ax.plot([-400, 50], [-400, 50], 'r--', lw=1.3, zorder=10)

    ax.set_title(f"{i+1}h prediction \n (CC={cc:.2f},  RMSE={rmse:.2f} nT)")
    ax.set_ylim(-400, 50)
    ax.set_xlim(-400, 50)


fig.supxlabel("Dst[nT]", fontsize=14, fontweight='bold')
fig.supylabel("DL-input Dst [nT]", fontsize=14, fontweight='bold')
plt.suptitle("Dst Prediction Comparision", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
dst_persis_1h = dst_persis_list[:, :-1].reshape((-1, dst_persis_list.shape[-1]//2, 2)).mean(axis=-1)

for i in range(12):
    ax = axes.flat[i]

    x = tar_dst[:, i]
    y = dst_persis_1h[:, i]

    mask = np.isfinite(x) & np.isfinite(y)
    cc = pearsonr(x[mask], y[mask])[0]
    rmse = np.mean(np.sqrt((x[mask] - y[mask]) ** 2))
    ax.scatter(
        x,
        y,
        color='k',
        marker='D',
        s=20
    )

    ax.plot([-400, 50], [-400, 50], 'r--', lw=1.3, zorder=10)

    ax.set_title(f"{i+1}h prediction \n (CC={cc:.2f},  RMSE={rmse:.2f} nT)")
    ax.set_ylim(-400, 50)
    ax.set_xlim(-400, 50)

fig.supxlabel("Dst[nT]", fontsize=14, fontweight='bold')
fig.supylabel("Persistent-input Dst [nT]", fontsize=14, fontweight='bold')
plt.suptitle("Dst Prediction Comparision", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
dst_preds_1h = dst_preds_list[:, :-1].reshape((-1, dst_preds_list.shape[-1]//2, 2)).mean(axis=-1)

for i in range(12):
    ax = axes.flat[i]

    x = dst_om_list[:, i]
    y = dst_preds_1h[:, i]

    mask = np.isfinite(x) & np.isfinite(y)
    cc = pearsonr(x[mask], y[mask])[0]
    ax.scatter(
        x,
        y,
        color='k',
        marker='D',
        s=20
    )

    ax.plot([-400, 50], [-400, 50], 'r--', lw=1.3, zorder=10)

    ax.set_title(f"{i+1}h prediction (CC={cc:.2f})")
    ax.set_ylim(-400, 50)
    ax.set_xlim(-400, 50)


fig.supxlabel("OM Dst[nT]", fontsize=14, fontweight='bold')
fig.supylabel("DL-input Dst [nT]", fontsize=14, fontweight='bold')
plt.suptitle("Dst Prediction Comparision Using O'Brien-McPherron Model", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
dst_persis_1h = dst_persis_list[:, :-1].reshape((-1, dst_persis_list.shape[-1]//2, 2)).mean(axis=-1)

for i in range(12):
    ax = axes.flat[i]

    x = dst_om_list[:, i]
    y = dst_persis_1h[:, i]

    mask = np.isfinite(x) & np.isfinite(y)
    cc = pearsonr(x[mask], y[mask])[0]
    ax.scatter(
        x,
        y,
        color='k',
        marker='D',
        s=20
    )

    ax.plot([-400, 50], [-400, 50], 'r--', lw=1.3, zorder=10)

    ax.set_title(f"{i+1}h prediction (CC={cc:.2f})")
    ax.set_ylim(-400, 50)
    ax.set_xlim(-400, 50)


fig.supxlabel("OM Dst[nT]", fontsize=14, fontweight='bold')
fig.supylabel("Persistent-input Dst [nT]", fontsize=14, fontweight='bold')
plt.suptitle("Dst Prediction Comparision Using O'Brien-McPherron Model", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
dst1 = omni[datetime.datetime(2024, 3, 3, 11, 0):datetime.datetime(2024, 3, 4, 0, 0)]['DST'].values
v1 = omni[datetime.datetime(2024, 3, 3, 11, 0):datetime.datetime(2024, 3, 4, 0, 0)]['V'].values
bz1 = omni[datetime.datetime(2024, 3, 3, 11, 0):datetime.datetime(2024, 3, 4, 0, 0)]['BZ_GSM'].values
n1 = omni[datetime.datetime(2024, 3, 3, 11, 0):datetime.datetime(2024, 3, 4, 0, 0)]['N'].values

In [ ]:
i = 774
plt.figure(figsize=(8, 5), facecolor='white')
x = list(inp[i, -6:, 1]) + [np.nan]*6
plt.plot(x, color='black', label='Input')

aa = [np.nan]*5 + [inp[i, -1, 1]]

ymin = -45
ymax = 15
plt.plot(aa+list(tar[i]), label='Target', color='gray', linewidth=2, marker='o', markersize=5)
plt.plot(aa+list(pred[i]), label='Pred', color='forestgreen', linestyle='--', linewidth=2, marker='o', markersize=5)
plt.vlines(x=5, ymin=-100, ymax=100, linestyles='dashed', color='red')
plt.title(f"Prediction time: {str(pd.to_datetime(tar_t[i][0])-pd.Timedelta(minutes=30))[:-3]} UTC", fontsize=12, weight='bold')
plt.ylim(ymin, ymax)
plt.legend()

plt.ylabel("Bz [nT]", fontsize=12, weight='bold')
plt.xlabel("Prediction time", fontsize=12, weight='bold')
plt.xticks(ticks=np.arange(1, 30, 4), labels=["-2h" , ""] + [f"+{h}h" for h in range(2, 13, 2)])
plt.grid()
plt.show()

In [ ]:

time = np.arange(0, 12, 0.5)
v = np.full_like(time, inp[i, -1, 2])
n = np.full_like(time, inp[i, -1, 3])
bz = pred[i]
dst_pred = dst_eqs.burton_dst(time, v, bz, n, dst1[0])

In [ ]:
time = np.arange(0, 12, 0.5)
v = np.full_like(time, inp[i, -1, 2])
n = np.full_like(time, inp[i, -1, 3])
bz = np.full_like(time, inp[i, -1, 1])
dst_persis = dst_eqs.burton_dst(time, v, bz, n, dst1[0])

In [ ]:
v1.shape

In [ ]:
time = np.arange(0, 12, 1)
v = v1[1:13]
n = n1[1:13]
bz = bz1[1:13]
dst_burton = dst_eqs.burton_dst(time, v, bz, n, dst1[0])

In [ ]:
dst1.shape

In [ ]:
plt.plot(np.arange(12), dst1[1:-1], label="Dst", marker='D', markersize=5)
plt.plot(np.arange(24)*0.5, dst_pred, label="DL prediction", marker='o', markersize=5)
plt.plot(np.arange(24)*0.5, dst_persis, label="Persistent", marker='*', markersize=5)
plt.plot(np.arange(12), dst_burton, label="Burton", marker='^', markersize=5)
plt.legend()
plt.xlabel("Time")
plt.ylabel("Dst [nT]")
plt.show()

In [ ]:
dst1 = omni[datetime.datetime(2024, 10, 10, 10, 0):datetime.datetime(2024, 10, 11, 18, 0)]['DST'].values
v1 = omni[datetime.datetime(2024, 10, 10, 10, 0):datetime.datetime(2024, 10, 11, 18, 0)]['V'].values
bz1 = omni[datetime.datetime(2024, 10, 10, 10, 0):datetime.datetime(2024, 10, 11, 18, 0)]['BZ_GSM'].values
n1 = omni[datetime.datetime(2024, 10, 10, 10, 0):datetime.datetime(2024, 10, 11, 18, 0)]['N'].values

In [ ]:
tar_t[951][0]

In [ ]:
indices = range(951, 955)
dst_preds_list = []
dst_persis_list = []
dst_burton_list = []
dst_om_list = []

for i, idx in enumerate(indices):
    time = np.arange(0, 12, 0.5)
    v = np.full_like(time, inp[idx, -1, 2])
    n = np.full_like(time, inp[idx, -1, 3])
    bz = pred[idx]
    dst_pred = dst_eqs.burton_dst(time, v, bz, n, dst1[9+i])
    dst_preds_list.append(dst_pred)

    time = np.arange(0, 12, 0.5)
    v = np.full_like(time, inp[idx, -1, 2])
    n = np.full_like(time, inp[idx, -1, 3])
    bz = np.full_like(time, inp[idx, -1, 1])
    dst_persis = dst_eqs.burton_dst(time, v, bz, n, dst1[9+i])
    dst_persis_list.append(dst_persis)

    time = np.arange(0, 12, 1)
    v = v1[10+i:22+i]
    n = n1[10+i:22+i]
    bz = bz1[10+i:22+i]
    dst_burton = dst_eqs.burton_dst(time, v, bz, n, dst1[9+i])
    dst_burton_list.append(dst_burton)

    time = np.arange(0, 12, 1)
    v = v1[10+i:22+i]
    n = n1[10+i:22+i]
    bz = bz1[10+i:22+i]
    dst_om = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, dst1[9+i])
    dst_om_list.append(dst_om)



In [ ]:
import matplotlib
colors = plt.colormaps['Set1']

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(np.arange(33), dst1, label="Dst", marker='D', markersize=5, color='k', alpha=0.5)
for i in range(len(indices)):
    plt.plot(np.arange(24)*0.5+(9.5 + 0.5*i), dst_preds_list[i], label="DL prediction", marker='o', markersize=5, c=colors(i))

plt.subplot(3, 1, 2)
plt.plot(np.arange(33), dst1, label="Dst", marker='D', markersize=5, color='k', alpha=0.5)
for i in range(len(indices)):
    plt.plot(np.arange(24)*0.5+(9.5 + 0.5*i), dst_persis_list[i], label="Persistent", marker='*', markersize=5, c=colors(i))

plt.subplot(3, 1, 3)
plt.plot(np.arange(33), dst1, label="Dst", marker='D', markersize=5, color='k', alpha=0.5)
for i in range(len(indices)):
    plt.plot(np.arange(12)+(10+i), dst_burton_list[i], label="Burton", marker='^', markersize=5, c=colors(i))

#plt.legend()
plt.xlabel("Time")
plt.ylabel("Dst [nT]")
plt.show()

In [ ]:
indices = range(951, 955)
dst_preds_list = []
dst_persis_list = []
dst_om_list = []

for i, idx in enumerate(indices):
    time = np.arange(0, 12, 0.5)
    v = np.full_like(time, inp[idx, -1, 2])
    n = np.full_like(time, inp[idx, -1, 3])
    bz = pred[idx]
    dst_pred = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, dst1[9+i])
    dst_preds_list.append(dst_pred)

    time = np.arange(0, 12, 0.5)
    v = np.full_like(time, inp[idx, -1, 2])
    n = np.full_like(time, inp[idx, -1, 3])
    bz = np.full_like(time, inp[idx, -1, 1])
    dst_persis = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, dst1[9+i])
    dst_persis_list.append(dst_persis)

    time = np.arange(0, 12, 1)
    v = v1[10+i:22+i]
    n = n1[10+i:22+i]
    bz = bz1[10+i:22+i]
    dst_om = dst_eqs.obrien_mcpherron_dst(time, v, n, bz, dst1[9+i])
    dst_om_list.append(dst_om)



In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(np.arange(33), dst1, label="Dst", marker='D', markersize=5, color='k', alpha=0.5)
for i in range(len(indices)):
    plt.plot(np.arange(24)*0.5+(9.5 + 0.5*i), dst_preds_list[i], label="DL prediction", marker='o', markersize=5, c=colors(i))

plt.subplot(3, 1, 2)
plt.plot(np.arange(33), dst1, label="Dst", marker='D', markersize=5, color='k', alpha=0.5)
for i in range(len(indices)):
    plt.plot(np.arange(24)*0.5+(9.5 + 0.5*i), dst_persis_list[i], label="Persistent", marker='*', markersize=5, c=colors(i))

plt.subplot(3, 1, 3)
plt.plot(np.arange(33), dst1, label="Dst", marker='D', markersize=5, color='k', alpha=0.5)
for i in range(len(indices)):
    plt.plot(np.arange(12)+(10+i), dst_om_list[i], label="Burton", marker='^', markersize=5, c=colors(i))

#plt.legend()
plt.xlabel("Time")
plt.ylabel("Dst [nT]")
plt.show()


In [ ]:
i = 951
plt.figure(figsize=(8, 4), facecolor='white')

aa = [np.nan]*5 + [inp[i, -1, 1]]

ymin = -50
ymax = 15
plt.plot(list(inp[i, -6:, 1]) +list(tar[i]), label='Target Bz', color='dimgray', linewidth=2, marker='D', markersize=5)
plt.plot(aa+list(pred[i]), label='Predicted Bz', color='blue', linestyle='--', linewidth=2, marker='o', markersize=5)
plt.vlines(x=5, ymin=-100, ymax=100, linestyles='dashed', color='red')
plt.title(f"Prediction time: {str(pd.to_datetime(tar_t[i][0])-pd.Timedelta(hours=1))[:-3]} UTC", fontsize=12, weight='bold')
plt.ylim(ymin, ymax)
plt.legend(fontsize=12)

plt.ylabel("Bz [nT]", fontsize=12, weight='bold')
plt.xlabel("Time", fontsize=12, weight='bold')
plt.xticks(ticks=np.arange(1, 30, 4), labels=["-2h" , ""] + [f"+{h}h" for h in range(2, 13, 2)])
plt.grid()
plt.show()

In [ ]:
len(np.arange(25)*0.5+(4))

In [ ]:
i = 951
tar_window_omni1 = omni[tar_t[i][0]-datetime.timedelta(hours=5):tar_t[i][-1]+datetime.timedelta(hours=5)]
tar_window_dst1 = tar_window_omni1['DST'].values


plt.figure(figsize=(8, 4))
ax = plt.gca()
plt.plot(np.arange(22), tar_window_dst1, label="Dst (Ground Truth)", marker='D', markersize=5, color='dimgray', alpha=1, linewidth=2)
plt.plot(np.arange(26)*0.5+(4), [tar_window_dst1[4]]+list(dst_preds_list[i]), label="O-M from DL prediction", marker='o', markersize=5, c='blue')
plt.plot(np.arange(26)*0.5+(4), [tar_window_dst1[4]]+list(dst_persis_list[i]), label="O-M from Persistent", marker='*', markersize=8, c='forestgreen')
plt.plot(np.arange(14)+4, [tar_window_dst1[4]]+list(dst_om_list[i]), label="O-M from OMNI", marker='^', markersize=6, c='red')
plt.axvspan(4, 16.5, alpha=0.2, color='pink')
plt.vlines(x=4, ymin=-400, ymax=200, color='crimson', linestyle='dashed')
plt.vlines(x=16.5, ymin=-400, ymax=200, color='crimson', linestyle='dashed')
plt.xlabel("Time")

plt.title(f"Prediction time: {str(tar_t[i][0]-datetime.timedelta(hours=1))[:-3]} UTC", fontsize=15, fontweight='bold')
plt.legend(loc='lower left')
plt.xticks([5, 7, 10, 13, 16], labels=['+1h', '+3h', '+6h', '+9h', '+12h'])

ax.xaxis.set_minor_locator(FixedLocator(np.arange(0, 25, 1)))
ax.tick_params(axis='x', which='minor', length=0)

plt.grid(True, which='major', axis='y', alpha=0.5)
plt.grid(True, which='minor', axis='x', alpha=0.5)
plt.grid(True, which='major', axis='x', alpha=0.5)

plt.xlabel("Time", fontsize=12, fontweight='bold')
plt.ylabel("Dst [nT]", fontsize=12, fontweight='bold')
plt.ylim(-350, 60)
plt.show()



In [ ]:

i = 951
tar_window_omni1 = omni[tar_t[i][0]-datetime.timedelta(hours=5):tar_t[i][-1]+datetime.timedelta(hours=5)]
tar_window_dst1 = tar_window_omni1['DST'].values


plt.figure(figsize=(8, 4))
plt.plot(np.arange(22), tar_window_dst1, label="Dst (Ground Truth)", marker='D', markersize=5, color='k', alpha=0.5)
plt.plot(np.arange(25)*0.5+(4), [tar_window_dst1[4]]+list(dst_preds_list[i]), label="O-M from DL prediction", marker='o', markersize=5, c=colors(0))
plt.plot(np.arange(25)*0.5+(4), [tar_window_dst1[4]]+list(dst_persis_list[i]), label="O-M from Persistent", marker='*', markersize=5, c=colors(1))
plt.plot(np.arange(13)+(4), [tar_window_dst1[4]]+list(dst_om_list[i]), label="O-M from OMNI", marker='^', markersize=5, c=colors(2))

plt.legend()
plt.xlabel("Time")
plt.ylabel("Dst [nT]")
plt.show()


In [ ]:
np.array(dst_preds_list).shape

In [ ]:
dst_om_list[0]

In [ ]:
plt.scatter(dst_om_list[0], np.array(dst_preds_list)[0, ::2], marker='D')
plt.scatter(dst_om_list[1], np.array(dst_preds_list)[1, ::2], marker='*')
plt.scatter(dst_om_list[2], np.array(dst_preds_list)[2, ::2], marker='^')
plt.scatter(dst_om_list[3], np.array(dst_preds_list)[3, ::2], marker='o')
plt.xlabel("O'brein & Mcpheeron", fontsize=12)
plt.ylabel("DL model", fontsize=12)
plt.show()